# 05 — Feature Engineering v2

**Goal:** Build the final feature matrix for the Top-5 Country Ranker model.

**Prediction task:** Given a song on its first chart day, which 5 countries will it chart in next (within 60 days)?

**Row structure:** One row per `(track_id, target_country)` — single observation at first chart date, with ~101 model features.

**Temporal split:** Train ≤ 2019 | Val 2020 | Test 2021 (by track's first chart date).

In [ ]:
## ── Cell 1: Setup & Imports ──────────────────────────────────────────────────

import duckdb
import pathlib
import os

# Paths
ROOT = pathlib.Path(os.getcwd()).parent  # repo root
V2_PATH = ROOT / "datasets" / "processed" / "v2" / "full"
AUX_PATH = ROOT / "datasets" / "processed" / "v1_aux"
OUT_PATH = ROOT / "datasets" / "processed" / "v3_features"
OUT_PATH.mkdir(parents=True, exist_ok=True)

# DuckDB — disk-backed for memory safety with 25M+ rows
DB_FILE = str(OUT_PATH / "_feature_eng.duckdb")
con = duckdb.connect(DB_FILE)
con.execute("SET threads = 4")
con.execute("SET memory_limit = '4GB'")

# Load v2 dataset as a view
con.execute(f"""
    CREATE OR REPLACE VIEW v2 AS
    SELECT * FROM read_parquet('{V2_PATH}/year=*/data_0.parquet', hive_partitioning=true)
""")

# Load auxiliary tables
con.execute(f"""
    CREATE OR REPLACE VIEW cultural_distance AS
    SELECT * FROM read_parquet('{AUX_PATH}/cultural_distance_long.parquet')
""")
con.execute(f"""
    CREATE OR REPLACE VIEW cultural_top5 AS
    SELECT * FROM read_parquet('{AUX_PATH}/cultural_distance_top5.parquet')
""")
con.execute(f"""
    CREATE OR REPLACE VIEW countries_ref AS
    SELECT * FROM read_parquet('{AUX_PATH}/countries_reference_clean.parquet')
""")

# Quick sanity check
for name in ['v2', 'cultural_distance', 'cultural_top5', 'countries_ref']:
    cnt = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name}: {cnt:,} rows")

In [ ]:
## ── Cell 2: Country List & Region Normalization ─────────────────────────────

import re

# Extract the 66 distinct countries from v2
countries = con.execute("SELECT DISTINCT region FROM v2 ORDER BY region").fetchdf()['region'].tolist()
print(f"Number of countries: {len(countries)}")

# Snake-case mapping for column names: "United States" → "united_states"
def to_snake(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

country_to_col = {c: f"rank_{to_snake(c)}" for c in countries}

# Register as a DuckDB table for later joins
con.execute("CREATE OR REPLACE TABLE country_list AS SELECT UNNEST($1::VARCHAR[]) AS country", [countries])

# Show a few examples
for c in countries[:5]:
    print(f"  {c:30s} → {country_to_col[c]}")
print(f"  ... ({len(countries)} total)")

In [ ]:
## ── Cell 3: Track-Level Table ────────────────────────────────────────────────
# One row per track_id (~195K tracks) with static attributes:
#   first_chart_date, audio features, release_date, explicit, duration_ms, artist
# first_chart_date is based on top200 entries only.

con.execute("""
    CREATE OR REPLACE TABLE tracks AS
    WITH first_appearance AS (
        SELECT
            track_id,
            MIN(date) AS first_chart_date,
            -- Take audio features & metadata from the first chart appearance
            FIRST(artist ORDER BY date, region)        AS artist,
            FIRST(release_date ORDER BY date, region)  AS release_date,
            FIRST(duration_ms ORDER BY date, region)   AS duration_ms,
            FIRST(explicit ORDER BY date, region)       AS explicit,
            FIRST(af_danceability ORDER BY date, region)     AS af_danceability,
            FIRST(af_energy ORDER BY date, region)           AS af_energy,
            FIRST(af_valence ORDER BY date, region)          AS af_valence,
            FIRST(af_tempo ORDER BY date, region)            AS af_tempo,
            FIRST(af_acousticness ORDER BY date, region)     AS af_acousticness,
            FIRST(af_speechiness ORDER BY date, region)      AS af_speechiness,
            FIRST(af_instrumentalness ORDER BY date, region) AS af_instrumentalness,
            FIRST(af_liveness ORDER BY date, region)         AS af_liveness
        FROM v2
        WHERE chart = 'top200'
        GROUP BY track_id
    )
    SELECT
        *,
        -- is_friday_release: release_date falls on a Friday (dayofweek=5 in DuckDB, 0=Mon)
        CASE WHEN dayofweek(release_date) = 4 THEN 1 ELSE 0 END AS is_friday_release
    FROM first_appearance
""")

cnt = con.execute("SELECT COUNT(*) FROM tracks").fetchone()[0]
print(f"Track table: {cnt:,} tracks")
con.execute("SELECT * FROM tracks LIMIT 3").fetchdf()

In [ ]:
## ── Cell 4: Observation Points ───────────────────────────────────────────────
# Single observation per track: the first day it charts in ANY country.
# Then we look at what happens in the next 60 days.

con.execute("""
    CREATE OR REPLACE TABLE observation_points AS
    SELECT track_id, first_chart_date AS observation_time
    FROM tracks
""")

obs_cnt = con.execute("SELECT COUNT(*) FROM observation_points").fetchone()[0]
print(f"Observation points: {obs_cnt:,} (one per track)")

# Sanity: should equal number of tracks
tracks_cnt = con.execute("SELECT COUNT(*) FROM tracks").fetchone()[0]
assert obs_cnt == tracks_cnt, f"Mismatch: {obs_cnt} obs vs {tracks_cnt} tracks"
print("Confirmed: exactly one observation per track.")

In [ ]:
## ── Cell 5: Rank Columns (Pivot) + Viral50 Flag ─────────────────────────────
# For each (track_id, observation_time), pivot top200 chart data into 66 rank columns.
# rank_XX = rank in country XX on the observation_time date (top200 only), or 0 if not charted.
# Also compute track_in_viral50_at_obs: 1 if the track appears on ANY viral50 chart on that day.

# Build the PIVOT SQL dynamically for all 66 countries
pivot_cases = ",\n        ".join([
    f"COALESCE(MAX(CASE WHEN region = '{c}' THEN rank END), 0) AS {country_to_col[c]}"
    for c in countries
])

con.execute(f"""
    CREATE OR REPLACE TABLE obs_with_ranks AS
    WITH top200_ranks AS (
        SELECT
            op.track_id,
            op.observation_time,
            {pivot_cases}
        FROM observation_points op
        LEFT JOIN v2
            ON v2.track_id = op.track_id
            AND v2.date = op.observation_time
            AND v2.chart = 'top200'
        GROUP BY op.track_id, op.observation_time
    ),
    viral_flag AS (
        SELECT
            op.track_id,
            op.observation_time,
            MAX(1) AS track_in_viral50_at_obs
        FROM observation_points op
        JOIN v2
            ON v2.track_id = op.track_id
            AND v2.date = op.observation_time
            AND v2.chart = 'viral50'
        GROUP BY op.track_id, op.observation_time
    )
    SELECT
        t.*,
        COALESCE(v.track_in_viral50_at_obs, 0) AS track_in_viral50_at_obs
    FROM top200_ranks t
    LEFT JOIN viral_flag v
        ON v.track_id = t.track_id AND v.observation_time = t.observation_time
""")

cnt = con.execute("SELECT COUNT(*) FROM obs_with_ranks").fetchone()[0]
cols = con.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'obs_with_ranks'").fetchone()[0]
viral_cnt = con.execute("SELECT SUM(track_in_viral50_at_obs) FROM obs_with_ranks").fetchone()[0]
print(f"obs_with_ranks: {cnt:,} rows × {cols} columns (2 ids + {cols - 3} rank cols + viral50 flag)")
print(f"Tracks on viral50 at observation time: {viral_cnt:,} ({viral_cnt/cnt*100:.1f}%)")

# Spot check: show a row with some non-zero ranks
con.execute("""
    SELECT * FROM obs_with_ranks
    WHERE rank_united_states > 0 AND rank_brazil > 0
    LIMIT 1
""").fetchdf()

In [ ]:
## ── Cell 6: Artist Features ──────────────────────────────────────────────────
# All computed using only top200 data with date < observation_time to prevent leakage.
# 7 features: prior_chart_count, prior_unique_regions, prior_best_rank,
#              prior_unique_tracks, multi_artist_flag, country_ratio,
#              prior_success_in_target (added later in cross-join cell)

# First, compute the multi_artist_flag from the artist string (static per track)
con.execute("""
    CREATE OR REPLACE TABLE track_artist_flag AS
    SELECT
        track_id,
        artist,
        CASE WHEN artist LIKE '%feat.%'
             OR artist LIKE '%Feat.%'
             OR artist LIKE '%, %'
             OR artist LIKE '% & %'
             OR artist LIKE '% x %'
             OR artist LIKE '% X %'
        THEN 1 ELSE 0 END AS multi_artist_flag
    FROM tracks
""")

# Artist-level prior features at each observation_time (top200 only)
con.execute("""
    CREATE OR REPLACE TABLE artist_features AS
    WITH prior_data AS (
        -- All top200 chart entries by the same artist BEFORE the observation_time
        SELECT
            op.track_id AS obs_track_id,
            op.observation_time,
            t.artist,
            v2.track_id AS prior_track_id,
            v2.region,
            v2.rank AS chart_rank,
            v2.date
        FROM observation_points op
        JOIN tracks t ON op.track_id = t.track_id
        JOIN v2 ON v2.artist = t.artist
              AND v2.date < op.observation_time
              AND v2.chart = 'top200'
    ),
    agg AS (
        SELECT
            obs_track_id AS track_id,
            observation_time,
            COUNT(*) AS artist_prior_chart_count,
            COUNT(DISTINCT region) AS artist_prior_unique_regions,
            MIN(chart_rank) AS artist_prior_best_rank,
            COUNT(DISTINCT prior_track_id) AS artist_prior_unique_tracks
        FROM prior_data
        GROUP BY obs_track_id, observation_time
    )
    SELECT
        a.track_id,
        a.observation_time,
        COALESCE(a.artist_prior_chart_count, 0) AS artist_prior_chart_count,
        COALESCE(a.artist_prior_unique_regions, 0) AS artist_prior_unique_regions,
        COALESCE(a.artist_prior_best_rank, 201) AS artist_prior_best_rank,
        COALESCE(a.artist_prior_unique_tracks, 0) AS artist_prior_unique_tracks,
        -- country_ratio = unique_regions / unique_tracks (0 if no prior tracks)
        CASE WHEN COALESCE(a.artist_prior_unique_tracks, 0) = 0 THEN 0.0
             ELSE a.artist_prior_unique_regions * 1.0 / a.artist_prior_unique_tracks
        END AS artist_country_ratio
    FROM agg a
""")

cnt = con.execute("SELECT COUNT(*) FROM artist_features").fetchone()[0]
print(f"Artist features computed for {cnt:,} (track, observation_time) pairs")
con.execute("SELECT * FROM artist_features LIMIT 3").fetchdf()

In [ ]:
## ── Cell 7: Target Cross-Join ────────────────────────────────────────────────
# For each (track_id, observation_time), cross-join with all 66 countries.
# Exclude countries where the track is already charting on top200 (rank > 0).
# Also compute artist_prior_success_in_target here (target-specific, top200 only).

# Build the condition to check if rank > 0 in any country
rank_cols = list(country_to_col.values())

# We need a reverse mapping: column name → country name
col_to_country = {v: k for k, v in country_to_col.items()}

# Build CASE expression: for each target country, check if its rank column > 0
filter_cases = " OR ".join([
    f"(cl.country = '{c}' AND owr.{country_to_col[c]} > 0)"
    for c in countries
])

con.execute(f"""
    CREATE OR REPLACE TABLE pairs AS
    SELECT
        owr.*,
        cl.country AS target_country
    FROM obs_with_ranks owr
    CROSS JOIN country_list cl
    -- Exclude countries where track is already charting on top200
    WHERE NOT ({filter_cases})
""")

cnt = con.execute("SELECT COUNT(*) FROM pairs").fetchone()[0]
print(f"Pair-level rows (after excluding already-charting): {cnt:,}")

# Now add artist_prior_success_in_target (top200 only)
con.execute("""
    CREATE OR REPLACE TABLE artist_target_success AS
    SELECT
        op.track_id,
        op.observation_time,
        v2.region AS target_country,
        COUNT(DISTINCT v2.track_id) AS artist_prior_success_in_target
    FROM observation_points op
    JOIN tracks t ON op.track_id = t.track_id
    JOIN v2 ON v2.artist = t.artist
              AND v2.date < op.observation_time
              AND v2.track_id != op.track_id
              AND v2.chart = 'top200'
    GROUP BY op.track_id, op.observation_time, v2.region
""")

cnt2 = con.execute("SELECT COUNT(*) FROM artist_target_success").fetchone()[0]
print(f"Artist-target success entries: {cnt2:,}")

In [ ]:
## ── Cell 8: Target Country Features ──────────────────────────────────────────
# Join country metadata (population, continent) from countries_reference_clean.
# Compute target_avg_daily_streams and target_new_entry_rate_30d from trailing 30-day windows.
# Uses top200 data only (streams are NULL for viral50 anyway).

# Optimize for laptop runs: compute reusable (date, country) rolling stats once,
# then expand back to the required (track, observation_time, target_country) grain.
con.execute("""
    CREATE OR REPLACE TABLE target_country_stats AS
    WITH observation_calendar AS (
        SELECT observation_time
        FROM generate_series(
            (SELECT MIN(observation_time) FROM observation_points),
            (SELECT MAX(observation_time) FROM observation_points),
            INTERVAL '1 day'
        ) AS t(observation_time)
    ),
    country_day_stats AS (
        SELECT
            region AS target_country,
            date AS stat_date,
            SUM(streams) AS total_streams,
            COUNT(*) AS chart_entries,
            SUM(CASE WHEN trend = 'NEW_ENTRY' THEN 1 ELSE 0 END) AS new_entries
        FROM v2
        WHERE chart = 'top200'
        GROUP BY region, date
    ),
    calendar_country_stats AS (
        SELECT
            oc.observation_time,
            cl.country AS target_country,
            COALESCE(cds.total_streams, 0) AS total_streams,
            COALESCE(cds.chart_entries, 0) AS chart_entries,
            COALESCE(cds.new_entries, 0) AS new_entries
        FROM observation_calendar oc
        CROSS JOIN country_list cl
        LEFT JOIN country_day_stats cds
               ON cds.target_country = cl.country
              AND cds.stat_date = oc.observation_time
    ),
    country_window_stats AS (
        SELECT
            observation_time,
            target_country,
            SUM(total_streams) OVER w * 1.0 / NULLIF(SUM(chart_entries) OVER w, 0) AS target_avg_daily_streams,
            SUM(new_entries) OVER w * 1.0 / NULLIF(SUM(chart_entries) OVER w, 0) AS target_new_entry_rate_30d
        FROM calendar_country_stats
        WINDOW w AS (
            PARTITION BY target_country
            ORDER BY observation_time
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        )
    )
    SELECT
        op.track_id,
        op.observation_time,
        cws.target_country,
        cws.target_avg_daily_streams,
        cws.target_new_entry_rate_30d
    FROM observation_points op
    JOIN country_window_stats cws
      ON cws.observation_time = op.observation_time
""")

cnt = con.execute("SELECT COUNT(*) FROM target_country_stats").fetchone()[0]
print(f"Target country stats: {cnt:,} track-country entries")

# Show sample
con.execute("""
    SELECT target_country, 
           ROUND(AVG(target_avg_daily_streams), 0) AS avg_streams,
           ROUND(AVG(target_new_entry_rate_30d), 4) AS avg_new_entry_rate
    FROM target_country_stats 
    GROUP BY target_country 
    ORDER BY avg_streams DESC 
    LIMIT 10
""").fetchdf()

In [ ]:
## ── Cell 9: Origin↔Target Relationship Features ─────────────────────────────
# 5 features: cultural_dist_min, cultural_dist_missing_flag,
#              same_language_flag, same_continent_flag, neighbor_entered_count

# We need to know which countries the track is currently charting in (origin set).
# This is encoded in the rank columns: rank > 0 means the track is in that country.

# Step 1: Build an "origin set" table — (track_id, observation_time, origin_country)
origin_unions = " UNION ALL ".join([
    f"SELECT track_id, observation_time, '{c}' AS origin_country FROM obs_with_ranks WHERE {country_to_col[c]} > 0"
    for c in countries
])

con.execute(f"""
    CREATE OR REPLACE TABLE origin_set AS
    {origin_unions}
""")

print(f"Origin set rows: {con.execute('SELECT COUNT(*) FROM origin_set').fetchone()[0]:,}")

# Step 2: cultural_dist_min — MIN distance from any origin country to the target
con.execute("""
    CREATE OR REPLACE TABLE cultural_dist_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        MIN(cd.cultural_distance) AS cultural_dist_min,
        CASE WHEN MIN(cd.cultural_distance) IS NULL THEN 1 ELSE 0 END AS cultural_dist_missing
    FROM pairs p
    JOIN origin_set os
        ON os.track_id = p.track_id AND os.observation_time = p.observation_time
    LEFT JOIN cultural_distance cd
        ON cd.source_country = os.origin_country AND cd.target_country = p.target_country
    GROUP BY p.track_id, p.observation_time, p.target_country
""")

# Impute missing cultural distances with the global median
median_dist = con.execute("""
    SELECT MEDIAN(cultural_dist_min) FROM cultural_dist_features WHERE cultural_dist_min IS NOT NULL
""").fetchone()[0]
print(f"Median cultural distance (for imputation): {median_dist:.3f}")

con.execute(f"""
    UPDATE cultural_dist_features
    SET cultural_dist_min = {median_dist}
    WHERE cultural_dist_min IS NULL
""")

# Step 3: same_language_flag — any origin country shares language with target
con.execute("""
    CREATE OR REPLACE TABLE language_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        MAX(CASE
            WHEN cr_origin.official_language IS NOT NULL
             AND cr_target.official_language IS NOT NULL
             AND (
                -- Check if any word in origin language appears in target language
                -- Simple approach: check if the primary language matches
                SPLIT_PART(cr_origin.official_language, ',', 1) = SPLIT_PART(cr_target.official_language, ',', 1)
                OR cr_origin.official_language LIKE '%' || SPLIT_PART(cr_target.official_language, ',', 1) || '%'
                OR cr_target.official_language LIKE '%' || SPLIT_PART(cr_origin.official_language, ',', 1) || '%'
             )
            THEN 1 ELSE 0
        END) AS same_language_flag,
        MAX(CASE
            WHEN cr_origin.continent = cr_target.continent THEN 1 ELSE 0
        END) AS same_continent_flag
    FROM pairs p
    JOIN origin_set os
        ON os.track_id = p.track_id AND os.observation_time = p.observation_time
    LEFT JOIN countries_ref cr_origin ON cr_origin.country = os.origin_country
    LEFT JOIN countries_ref cr_target ON cr_target.country = p.target_country
    GROUP BY p.track_id, p.observation_time, p.target_country
""")

# Step 4: neighbor_entered_count — of target's top-5 cultural neighbors,
#          how many has the track already entered?
con.execute("""
    CREATE OR REPLACE TABLE neighbor_features AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        COALESCE(SUM(CASE WHEN os.origin_country IS NOT NULL THEN 1 ELSE 0 END), 0)
            AS neighbor_entered_count
    FROM pairs p
    LEFT JOIN cultural_top5 ct5
        ON ct5.source_country = p.target_country
    LEFT JOIN origin_set os
        ON os.track_id = p.track_id
        AND os.observation_time = p.observation_time
        AND os.origin_country = ct5.target_country
    GROUP BY p.track_id, p.observation_time, p.target_country
""")

for tbl in ['cultural_dist_features', 'language_features', 'neighbor_features']:
    cnt = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {cnt:,} rows")

In [ ]:
## ── Cell 10: Labels ──────────────────────────────────────────────────────────
# did_enter_within_60d: 1 if track enters top200 in target within 60 days of observation_time
# days_to_entry: days until first top200 entry (NULL for negatives)

# First top200 chart date per (track_id, country)
con.execute("""
    CREATE OR REPLACE TABLE track_country_first_chart AS
    SELECT track_id, region AS country, MIN(date) AS first_chart_date
    FROM v2
    WHERE chart = 'top200'
    GROUP BY track_id, region
""")

con.execute("""
    CREATE OR REPLACE TABLE labels AS
    SELECT
        p.track_id,
        p.observation_time,
        p.target_country,
        CASE
            WHEN tcfc.first_chart_date IS NOT NULL
             AND tcfc.first_chart_date > p.observation_time
             AND tcfc.first_chart_date <= p.observation_time + INTERVAL '60 days'
            THEN 1 ELSE 0
        END AS did_enter_within_60d,
        CASE
            WHEN tcfc.first_chart_date IS NOT NULL
             AND tcfc.first_chart_date > p.observation_time
             AND tcfc.first_chart_date <= p.observation_time + INTERVAL '60 days'
            THEN (tcfc.first_chart_date - p.observation_time)::INTEGER
            ELSE NULL
        END AS days_to_entry
    FROM pairs p
    LEFT JOIN track_country_first_chart tcfc
        ON tcfc.track_id = p.track_id AND tcfc.country = p.target_country
""")

pos = con.execute("SELECT SUM(did_enter_within_60d) FROM labels").fetchone()[0]
total = con.execute("SELECT COUNT(*) FROM labels").fetchone()[0]
print(f"Labels: {total:,} rows, {pos:,} positives ({pos/total*100:.2f}%)")
print(f"Class ratio (neg:pos): {(total-pos)/pos:.1f}:1")

# days_to_entry distribution for positives
con.execute("""
    SELECT
        MIN(days_to_entry) AS min_days,
        ROUND(AVG(days_to_entry), 1) AS avg_days,
        MEDIAN(days_to_entry) AS median_days,
        MAX(days_to_entry) AS max_days
    FROM labels
    WHERE did_enter_within_60d = 1
""").fetchdf()

In [ ]:
## ── Cell 11: Temporal Features & Final Assembly ──────────────────────────────
# Combine ALL features into the final table.
# Add observation_month, observation_year, days_since_release, is_friday_release.
# One-hot encode target_country_continent.
# Include track_in_viral50_at_obs flag from Cell 5.

# Build the continent one-hot columns dynamically
continents = con.execute("""
    SELECT DISTINCT continent FROM countries_ref WHERE continent IS NOT NULL ORDER BY continent
""").fetchdf()['continent'].tolist()
print(f"Continents for one-hot: {continents}")

continent_cases = ",\n        ".join([
    f"CASE WHEN cr.continent = '{cont}' THEN 1 ELSE 0 END AS target_continent_{to_snake(cont)}"
    for cont in continents
])

# Build rank column list for SELECT
rank_col_list = ", ".join([f"p.{col}" for col in rank_cols])

con.execute(f"""
    CREATE OR REPLACE TABLE features_final AS
    SELECT
        -- Identifiers
        p.track_id,
        p.observation_time,
        p.target_country,

        -- A. Per-country rank columns (66, top200 only)
        {rank_col_list},

        -- B. Audio features (8)
        t.af_danceability,
        t.af_energy,
        t.af_valence,
        t.af_tempo,
        t.af_acousticness,
        t.af_speechiness,
        t.af_instrumentalness,
        t.af_liveness,

        -- C. Track metadata (5)
        t.duration_ms,
        CASE WHEN t.explicit THEN 1 ELSE 0 END AS explicit,
        CASE WHEN t.release_date IS NOT NULL
             THEN (p.observation_time - t.release_date)::INTEGER
             ELSE NULL
        END AS days_since_release,
        COALESCE(t.is_friday_release, 0) AS is_friday_release,
        p.track_in_viral50_at_obs,

        -- D. Artist features (7)
        COALESCE(af.artist_prior_chart_count, 0)    AS artist_prior_chart_count,
        COALESCE(af.artist_prior_unique_regions, 0)  AS artist_prior_unique_regions,
        COALESCE(af.artist_prior_best_rank, 201)     AS artist_prior_best_rank,
        COALESCE(af.artist_prior_unique_tracks, 0)   AS artist_prior_unique_tracks,
        COALESCE(taf.multi_artist_flag, 0)           AS multi_artist_flag,
        COALESCE(af.artist_country_ratio, 0.0)       AS artist_country_ratio,
        COALESCE(ats.artist_prior_success_in_target, 0) AS artist_prior_success_in_target,

        -- E. Target country features (4 + continent one-hot)
        COALESCE(cr.population, 0)                   AS target_population,
        COALESCE(tcs.target_avg_daily_streams, 0)    AS target_avg_daily_streams,
        COALESCE(tcs.target_new_entry_rate_30d, 0)   AS target_new_entry_rate_30d,
        {continent_cases},

        -- F. Origin↔Target relationship features (5)
        COALESCE(cdf.cultural_dist_min, {median_dist})  AS cultural_dist_min,
        COALESCE(cdf.cultural_dist_missing, 1)          AS cultural_dist_missing,
        COALESCE(lf.same_language_flag, 0)              AS same_language_flag,
        COALESCE(lf.same_continent_flag, 0)             AS same_continent_flag,
        COALESCE(nf.neighbor_entered_count, 0)          AS neighbor_entered_count,

        -- G. Temporal features (2)
        EXTRACT(MONTH FROM p.observation_time)::INTEGER AS observation_month,
        EXTRACT(YEAR FROM p.observation_time)::INTEGER  AS observation_year,

        -- Targets
        lb.did_enter_within_60d,
        lb.days_to_entry

    FROM pairs p
    JOIN tracks t ON t.track_id = p.track_id
    LEFT JOIN artist_features af
        ON af.track_id = p.track_id AND af.observation_time = p.observation_time
    LEFT JOIN track_artist_flag taf ON taf.track_id = p.track_id
    LEFT JOIN artist_target_success ats
        ON ats.track_id = p.track_id AND ats.observation_time = p.observation_time
        AND ats.target_country = p.target_country
    LEFT JOIN target_country_stats tcs
        ON tcs.track_id = p.track_id AND tcs.observation_time = p.observation_time
        AND tcs.target_country = p.target_country
    LEFT JOIN countries_ref cr ON cr.country = p.target_country
    LEFT JOIN cultural_dist_features cdf
        ON cdf.track_id = p.track_id AND cdf.observation_time = p.observation_time
        AND cdf.target_country = p.target_country
    LEFT JOIN language_features lf
        ON lf.track_id = p.track_id AND lf.observation_time = p.observation_time
        AND lf.target_country = p.target_country
    LEFT JOIN neighbor_features nf
        ON nf.track_id = p.track_id AND nf.observation_time = p.observation_time
        AND nf.target_country = p.target_country
    LEFT JOIN labels lb
        ON lb.track_id = p.track_id AND lb.observation_time = p.observation_time
        AND lb.target_country = p.target_country
""")

total = con.execute("SELECT COUNT(*) FROM features_final").fetchone()[0]
ncols = con.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'features_final'").fetchone()[0]
print(f"\nFinal feature table: {total:,} rows × {ncols} columns")

# Show column names
cols_df = con.execute("SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'features_final' ORDER BY ordinal_position").fetchdf()
print(f"\nAll columns ({len(cols_df)}):")
for _, row in cols_df.iterrows():
    print(f"  {row['column_name']:45s} {row['data_type']}")

In [ ]:
## ── Cell 12: Class Balance & Downsampling ────────────────────────────────────
# Report positive/negative ratio.
# Downsample negatives to 5:1 ratio within training set only (applied in Cell 13).

pos_rate = con.execute("""
    SELECT
        SUM(did_enter_within_60d) AS positives,
        COUNT(*) - SUM(did_enter_within_60d) AS negatives,
        COUNT(*) AS total,
        ROUND(SUM(did_enter_within_60d) * 100.0 / COUNT(*), 2) AS pos_pct,
        ROUND((COUNT(*) - SUM(did_enter_within_60d)) * 1.0 / NULLIF(SUM(did_enter_within_60d), 0), 1) AS neg_to_pos_ratio
    FROM features_final
""").fetchdf()
print("Class balance (full dataset):")
print(pos_rate.to_string(index=False))

# Breakdown by year
print("\nClass balance by observation_year:")
con.execute("""
    SELECT
        observation_year,
        SUM(did_enter_within_60d) AS positives,
        COUNT(*) AS total,
        ROUND(SUM(did_enter_within_60d) * 100.0 / COUNT(*), 2) AS pos_pct
    FROM features_final
    GROUP BY observation_year
    ORDER BY observation_year
""").fetchdf()

In [ ]:
## ── Cell 13: Train/Val/Test Split ────────────────────────────────────────────
# Train: first_chart_date ≤ 2019-12-31
# Val:   first_chart_date in 2020
# Test:  first_chart_date in 2021
# All rows of same track stay in same split (trivially true: one obs per track).
# Downsample negatives to 5:1 in training set only.

# Add first_chart_date to features_final for splitting
con.execute("""
    CREATE OR REPLACE TABLE features_with_split AS
    SELECT
        ff.*,
        t.first_chart_date,
        CASE
            WHEN t.first_chart_date <= '2019-12-31' THEN 'train'
            WHEN t.first_chart_date >= '2020-01-01' AND t.first_chart_date <= '2020-12-31' THEN 'val'
            WHEN t.first_chart_date >= '2021-01-01' THEN 'test'
        END AS split
    FROM features_final ff
    JOIN tracks t ON t.track_id = ff.track_id
""")

# Split stats before downsampling
print("Split stats (before downsampling):")
con.execute("""
    SELECT
        split,
        COUNT(*) AS rows,
        SUM(did_enter_within_60d) AS positives,
        ROUND(SUM(did_enter_within_60d) * 100.0 / COUNT(*), 2) AS pos_pct,
        COUNT(DISTINCT track_id) AS tracks
    FROM features_with_split
    GROUP BY split
    ORDER BY split
""").fetchdf()

# Downsample training negatives to 5:1 ratio
train_pos = con.execute("SELECT SUM(did_enter_within_60d) FROM features_with_split WHERE split = 'train'").fetchone()[0]
target_neg = train_pos * 5
train_neg_total = con.execute("SELECT COUNT(*) FROM features_with_split WHERE split = 'train' AND did_enter_within_60d = 0").fetchone()[0]

sample_frac = min(1.0, target_neg / train_neg_total) if train_neg_total > 0 else 1.0
print(f"\nTraining set: {train_pos:,} positives, {train_neg_total:,} negatives")
print(f"Downsampling negatives to {target_neg:,} (fraction: {sample_frac:.4f})")

# Use TABLESAMPLE with a seed for reproducibility
con.execute(f"""
    CREATE OR REPLACE TABLE train_set AS
    -- All positives
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split
    WHERE split = 'train' AND did_enter_within_60d = 1
    UNION ALL
    -- Downsampled negatives
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split
    WHERE split = 'train' AND did_enter_within_60d = 0
    USING SAMPLE {sample_frac * 100:.2f} PERCENT (bernoulli, 42)
""")

con.execute("""
    CREATE OR REPLACE TABLE val_set AS
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split WHERE split = 'val'
""")

con.execute("""
    CREATE OR REPLACE TABLE test_set AS
    SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split WHERE split = 'test'
""")

# Final split stats
for name in ['train_set', 'val_set', 'test_set']:
    total = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    pos = con.execute(f"SELECT SUM(did_enter_within_60d) FROM {name}").fetchone()[0]
    tracks = con.execute(f"SELECT COUNT(DISTINCT track_id) FROM {name}").fetchone()[0]
    print(f"\n{name}: {total:,} rows, {pos:,} positives ({pos/total*100:.2f}%), {tracks:,} tracks")

# Verify no track leakage between splits
leakage = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT track_id FROM train_set
        INTERSECT
        SELECT DISTINCT track_id FROM val_set
    )
""").fetchone()[0]
leakage2 = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT track_id FROM train_set
        INTERSECT
        SELECT DISTINCT track_id FROM test_set
    )
""").fetchone()[0]
leakage3 = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT track_id FROM val_set
        INTERSECT
        SELECT DISTINCT track_id FROM test_set
    )
""").fetchone()[0]
print(f"\nTrack leakage check: train∩val={leakage}, train∩test={leakage2}, val∩test={leakage3}")
assert leakage == 0 and leakage2 == 0 and leakage3 == 0, "LEAKAGE DETECTED!"
print("No track leakage between splits.")

In [ ]:
## ── Cell 14: Export ──────────────────────────────────────────────────────────
# Export train/val/test as separate parquet files.
# Log stats: row counts, positive rates, feature distributions.

import json
from datetime import datetime

export_path = OUT_PATH
print(f"Exporting to: {export_path}")

for split_name, table_name in [('train', 'train_set'), ('val', 'val_set'), ('test', 'test_set')]:
    out_file = export_path / f"{split_name}.parquet"
    con.execute(f"""
        COPY (SELECT * FROM {table_name})
        TO '{out_file}' (FORMAT PARQUET, COMPRESSION 'zstd')
    """)
    size_mb = out_file.stat().st_size / 1024 / 1024
    cnt = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  {split_name}.parquet: {cnt:,} rows, {size_mb:.1f} MB")

# Also export the full (non-downsampled) dataset for reference
full_file = export_path / "full.parquet"
con.execute(f"""
    COPY (SELECT * EXCLUDE (first_chart_date, split) FROM features_with_split)
    TO '{full_file}' (FORMAT PARQUET, COMPRESSION 'zstd')
""")
full_size = full_file.stat().st_size / 1024 / 1024
full_cnt = con.execute("SELECT COUNT(*) FROM features_with_split").fetchone()[0]
print(f"  full.parquet: {full_cnt:,} rows, {full_size:.1f} MB")

# Write manifest
manifest = {
    "created": datetime.now().isoformat(),
    "description": "Feature-engineered dataset for Top-5 Country Ranker (60-day horizon, first-chart-day observation)",
    "splits": {},
}
for split_name, table_name in [('train', 'train_set'), ('val', 'val_set'), ('test', 'test_set')]:
    cnt = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    pos = con.execute(f"SELECT SUM(did_enter_within_60d) FROM {table_name}").fetchone()[0]
    tracks = con.execute(f"SELECT COUNT(DISTINCT track_id) FROM {table_name}").fetchone()[0]
    manifest["splits"][split_name] = {
        "rows": cnt,
        "positives": pos,
        "positive_rate": round(pos / cnt * 100, 2),
        "tracks": tracks,
        "file": f"{split_name}.parquet",
    }

manifest_file = export_path / "manifest.json"
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"\nManifest written to {manifest_file}")
print(json.dumps(manifest, indent=2))

In [ ]:
## ── Cell 15: Verification ────────────────────────────────────────────────────
# 1. Schema check, value ranges, no NaN in critical columns
# 2. Verify temporal split correctness (no future leakage)
# 3. Spot-check rows

print("=" * 70)
print("VERIFICATION SUITE")
print("=" * 70)

# 1. Schema check — verify expected column count
expected_feature_cols = (
    66  # rank columns (top200)
    + 8  # audio
    + 5  # track metadata (duration, explicit, days_since_release, is_friday_release, viral50 flag)
    + 7  # artist
    + 3 + len(continents)  # target country (pop, streams, entry_rate + continent one-hots)
    + 5  # relationship
    + 2  # temporal
)
expected_total = 3 + expected_feature_cols + 2  # 3 identifiers + features + 2 targets
actual_cols = con.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'train_set'").fetchone()[0]
print(f"\n[1] Column count: expected ~{expected_total}, got {actual_cols}")

# 2. Value range checks
print("\n[2] Value range checks:")
checks = [
    ("Rank columns in [0, 200]",
     f"SELECT COUNT(*) FROM train_set WHERE {' OR '.join([f'{col} < 0 OR {col} > 200' for col in rank_cols[:10]])}"),
    ("Audio features in [0, 1] (except tempo)",
     "SELECT COUNT(*) FROM train_set WHERE af_danceability < 0 OR af_danceability > 1 OR af_energy < 0 OR af_energy > 1"),
    ("did_enter_within_60d in {0, 1}",
     "SELECT COUNT(*) FROM train_set WHERE did_enter_within_60d NOT IN (0, 1)"),
    ("days_to_entry in [1, 60] for positives",
     "SELECT COUNT(*) FROM train_set WHERE did_enter_within_60d = 1 AND (days_to_entry < 1 OR days_to_entry > 60)"),
    ("observation_month in [1, 12]",
     "SELECT COUNT(*) FROM train_set WHERE observation_month < 1 OR observation_month > 12"),
    ("track_in_viral50_at_obs in {0, 1}",
     "SELECT COUNT(*) FROM train_set WHERE track_in_viral50_at_obs NOT IN (0, 1)"),
]
all_passed = True
for desc, query in checks:
    bad = con.execute(query).fetchone()[0]
    status = "PASS" if bad == 0 else f"FAIL ({bad:,} violations)"
    if bad > 0:
        all_passed = False
    print(f"  {status}: {desc}")

# 3. No NaN/NULL in critical columns
print("\n[3] NULL checks on critical columns:")
critical_cols = ['track_id', 'observation_time', 'target_country', 'did_enter_within_60d',
                 'af_danceability', 'af_energy', 'cultural_dist_min', 'track_in_viral50_at_obs']
for col in critical_cols:
    nulls = con.execute(f"SELECT COUNT(*) FROM train_set WHERE {col} IS NULL").fetchone()[0]
    status = "PASS" if nulls == 0 else f"FAIL ({nulls:,} NULLs)"
    if nulls > 0:
        all_passed = False
    print(f"  {status}: {col}")

# 4. Temporal split — no future leakage
print("\n[4] Temporal split verification:")
for split_name, table_name in [('train', 'train_set'), ('val', 'val_set'), ('test', 'test_set')]:
    date_range = con.execute(f"""
        SELECT MIN(t.first_chart_date), MAX(t.first_chart_date)
        FROM {table_name} s JOIN tracks t ON t.track_id = s.track_id
    """).fetchone()
    print(f"  {split_name}: first_chart_date range [{date_range[0]} — {date_range[1]}]")

# 5. Spot-check: sample 3 rows from train set
print("\n[5] Sample rows (train set):")
sample = con.execute("""
    SELECT track_id, observation_time, target_country,
           track_in_viral50_at_obs, artist_prior_chart_count, cultural_dist_min,
           same_language_flag, neighbor_entered_count, did_enter_within_60d, days_to_entry
    FROM train_set
    WHERE did_enter_within_60d = 1
    LIMIT 3
""").fetchdf()
print(sample.to_string(index=False))

# 6. Feature distributions summary
print("\n[6] Feature distributions (train set):")
con.execute("""
    SELECT
        ROUND(AVG(cultural_dist_min), 3) AS avg_cult_dist,
        ROUND(AVG(same_language_flag), 3) AS avg_same_lang,
        ROUND(AVG(same_continent_flag), 3) AS avg_same_cont,
        ROUND(AVG(neighbor_entered_count), 3) AS avg_neighbor_cnt,
        ROUND(AVG(artist_prior_chart_count), 0) AS avg_artist_charts,
        ROUND(AVG(artist_prior_success_in_target), 2) AS avg_artist_target_success,
        ROUND(AVG(track_in_viral50_at_obs), 3) AS viral50_rate,
        ROUND(AVG(days_since_release), 0) AS avg_days_since_rel
    FROM train_set
""").fetchdf()

print("\n" + "=" * 70)
print(f"ALL CHECKS {'PASSED' if all_passed else 'SOME FAILED — review above'}")
print("=" * 70)

# Cleanup: close the DuckDB connection
# (the .duckdb file persists for re-runs; parquet exports are the deliverables)
con.close()
print("\nDuckDB connection closed. Feature engineering complete.")